# ROC Curve & AUC — Implementation Guide

> **Dataset:** Pima Indians Diabetes | **Models:** Logistic Regression vs SVM

---

## Core Concepts

**ROC Curve** — plots **True Positive Rate (TPR/Recall)** on the Y-axis vs **False Positive Rate (FPR)** on the X-axis by sweeping the classification threshold from high → low.
- **TPR** = of all actual positives, how many did we catch?
- **FPR** = of all actual negatives, how many did we falsely flag?

| Threshold | Effect |
|---|---|
| High (→ 1) | Cautious predictions → Low FPR, Low TPR → near **(0, 0)** |
| Low (→ 0) | Predict everything positive → near **(1, 1)** |
| **Optimal** | Top-left corner **(0, 1)** — perfect TPR with zero FPR |

**AUC (Area Under the ROC Curve)** — a single number summarising the curve.  
> *"The probability that the model scores a random positive higher than a random negative."*

| AUC | Meaning |
|---|---|
| 1.0 | Perfect classifier |
| 0.9–1.0 | Excellent |
| 0.8–0.9 | Good |
| 0.7–0.8 | Fair |
| 0.5 | Random — no skill |
| < 0.5 | Worse than random |

**Optimal Threshold (Youden's J)** — pick the threshold that maximises `TPR − FPR`. This gives the best balance between catching positives and avoiding false alarms.


## Setup

In [3]:
import pandas as pd

data = pd.read_csv('../../datasets/diabetes.csv')

data.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
X = data.drop('Outcome', axis=1)
y = data['Outcome']


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)


In [6]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Predicted probabilities for the positive class
y_scores = model.predict_proba(X_test)[:, 1]

# Compute ROC curve: sweep threshold from high → low
fpr, tpr, thresholds = roc_curve(y_test, y_scores)
roc_auc = roc_auc_score(y_test, y_scores)

print(f"AUC: {roc_auc:.4f}")
print(f"Number of threshold points: {len(thresholds)}")


## Optimal Threshold (Youden's J)

In [ ]:
import numpy as np

# Youden's J statistic: J = TPR - FPR → pick threshold that maximises this
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

print(f"Optimal threshold (Youden's J): {optimal_threshold:.4f}")
print(f"  TPR at optimal: {tpr[optimal_idx]:.4f}")
print(f"  FPR at optimal: {fpr[optimal_idx]:.4f}")


Optimal threshold is: 0.3686614815032027


## ROC Curve with AUC

In [ ]:
import plotly.graph_objects as go

# ROC curve
trace_roc = go.Scatter(x=fpr, y=tpr, mode='lines',
                        name=f'Logistic Regression (AUC = {roc_auc:.3f})',
                        line=dict(color='royalblue', width=2))

# Optimal threshold point
trace_opt = go.Scatter(x=[fpr[optimal_idx]], y=[tpr[optimal_idx]],
                        mode='markers+text',
                        name=f'Optimal θ = {optimal_threshold:.2f}',
                        marker=dict(color='red', size=10, symbol='star'),
                        text=[f'θ={optimal_threshold:.2f}'],
                        textposition='top right')

# Random baseline
trace_base = go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
                         name='Random (AUC = 0.5)',
                         line=dict(dash='dash', color='gray'))

fig = go.Figure(
    data=[trace_roc, trace_opt, trace_base],
    layout=go.Layout(
        title='ROC Curve — Logistic Regression',
        xaxis=dict(title='False Positive Rate'),
        yaxis=dict(title='True Positive Rate'),
        width=700, height=600,
        legend=dict(x=0.6, y=0.1)
    )
)
fig.show()


## Model Comparison — Logistic Regression vs SVM

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# SVM needs scaled features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# --- Logistic Regression ---
lr_fpr, lr_tpr, _ = roc_curve(y_test, y_scores)   # already computed above
lr_auc = roc_auc

# --- SVM ---
svm_model = SVC(probability=True)
svm_model.fit(X_train_scaled, y_train)
svm_scores = svm_model.predict_proba(X_test_scaled)[:, 1]
svm_fpr, svm_tpr, _ = roc_curve(y_test, svm_scores)
svm_auc = roc_auc_score(y_test, svm_scores)

fig = go.Figure(
    data=[
        go.Scatter(x=lr_fpr,  y=lr_tpr,  mode='lines',
                   name=f'Logistic Regression (AUC = {lr_auc:.3f})',
                   line=dict(color='royalblue', width=2)),
        go.Scatter(x=svm_fpr, y=svm_tpr, mode='lines',
                   name=f'SVM (AUC = {svm_auc:.3f})',
                   line=dict(color='darkorange', width=2)),
        go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
                   name='Random (AUC = 0.5)',
                   line=dict(dash='dash', color='gray')),
    ],
    layout=go.Layout(
        title='ROC Curve — Model Comparison',
        xaxis=dict(title='False Positive Rate'),
        yaxis=dict(title='True Positive Rate'),
        width=700, height=600,
        legend=dict(x=0.5, y=0.1)
    )
)
fig.show()

print(f"LR  AUC: {lr_auc:.4f}")
print(f"SVM AUC: {svm_auc:.4f}")
print(f"Winner : {'SVM' if svm_auc > lr_auc else 'Logistic Regression'}")


## Results & Interpretation

The model with the **higher AUC** ranks positives above negatives more reliably across all thresholds.  
> High AUC ≠ high accuracy at a fixed threshold — always pick the operating threshold based on business needs.

---

## Top 5 Interview Questions

**Q1. What is the ROC curve?**  
A graph of TPR (sensitivity) vs FPR (1 − specificity) at every possible classification threshold. It shows the trade-off between catching positives and generating false alarms.

---

**Q2. What does AUC represent intuitively?**  
The probability that a randomly chosen positive gets a higher predicted score than a randomly chosen negative. AUC = 1 is perfect; AUC = 0.5 is random guessing.

---

**Q3. What does AUC = 0.5 mean?**  
The model has zero discriminative power — no better than flipping a coin. Its ROC curve is a diagonal line.

---

**Q4. When should you use PR-AUC instead of ROC-AUC?**  
With severe class imbalance (e.g. fraud, rare disease). ROC-AUC can look artificially high because FPR is diluted by a massive pool of true negatives. PR-AUC focuses purely on the minority class performance.

---

**Q5. How do you choose the best classification threshold in production?**  
Use **Youden's J** (`TPR − FPR` is maximised) as a starting point, then adjust based on business cost — e.g. in medical diagnosis, missing a positive (FN) is far more costly than a false alarm (FP), so lower the threshold.

---

> **One-liner:** ROC-AUC measures ranking quality independent of threshold; AUC = 1 is perfect, 0.5 is random — prefer PR-AUC for imbalanced datasets.
